<a href="https://colab.research.google.com/github/raflyalvish-id/Customer-Behavior-Product-Association-Analysis/blob/main/Customer_Behavior_%26_Product_Association_Analysis_Rafly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Customer Behavior & Product Association Analysis**

In [ ]:
import pandas as pd
df = pd.read_csv('/content/online_ritel/online_retail_II.csv')

In [ ]:
print(df.head())

  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 176830 entries, 0 to 176829
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   Invoice      176830 non-null  object 
 1   StockCode    176830 non-null  object 
 2   Description  175518 non-null  object 
 3   Quantity     176830 non-null  int64  
 4   InvoiceDate  176830 non-null  object 
 5   Price        176830 non-null  float64
 6   Customer ID  135067 non-null  float64
 7   Country      176830 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 10.8+ MB


# Cleaning Data


## 1. Missing value

In [ ]:
df = df.dropna()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df = df[df['Quantity'] > 0]
df = df[df['Price'] > 0]

df['TotalPrice'] = df['Quantity'] * df['Price']

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 131534 entries, 0 to 176829
Data columns (total 9 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      131534 non-null  object        
 1   StockCode    131534 non-null  object        
 2   Description  131534 non-null  object        
 3   Quantity     131534 non-null  int64         
 4   InvoiceDate  131534 non-null  datetime64[ns]
 5   Price        131534 non-null  float64       
 6   Customer ID  131534 non-null  float64       
 7   Country      131534 non-null  object        
 8   TotalPrice   131534 non-null  float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(4)
memory usage: 10.0+ MB


In [ ]:
df['Customer ID'] = df['Customer ID'].astype(int)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID,TotalPrice
count,131534.000000,131534,131534.000000,131534.000000,131534.000000
mean,14.833686,2010-02-13 01:31:01.152249344,3.490880,15342.327330,22.539952
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-01-07 12:22:00,1.250000,13975.000000,5.040000
50%,5.000000,2010-02-17 11:27:00,1.950000,15270.000000,12.500000
75%,12.000000,2010-03-24 10:00:00,3.750000,16798.000000,19.500000
max,19152.000000,2010-04-28 10:04:00,10953.500000,18286.000000,10953.500000
std,135.168681,NaN,48.805091,1676.011544,77.225723


In [ ]:
df.isna().sum()

,0
Invoice,0
StockCode,0
Description,0
Quantity,0
InvoiceDate,0
Price,0
Customer ID,0
Country,0
TotalPrice,0


## 2. Duplicate

In [ ]:
df.duplicated().sum()

np.int64(1994)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

np.int64(0)

# RFM Analysis

In [ ]:
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (df['InvoiceDate'].max() - x.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']

In [ ]:
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,56,10,230.55
12358.0,141,1,1429.83
12359.0,36,4,1522.23
12360.0,34,2,158.00
12361.0,91,1,109.20


In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, duplicates='drop')
rfm['F_score'] = pd.qcut(rfm['Frequency'], 4, duplicates='drop')
rfm['M_score'] = pd.qcut(rfm['Monetary'], 4, duplicates='drop')

# paksa jadi category
rfm['R_score'] = rfm['R_score'].astype('category').cat.codes + 1
rfm['F_score'] = rfm['F_score'].astype('category').cat.codes + 1
rfm['M_score'] = rfm['M_score'].astype('category').cat.codes + 1

In [ ]:
rfm.dtypes

,0
Recency,int64
Frequency,int64
Monetary,float64
R_score,int8
F_score,int8
M_score,int8


In [ ]:
rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [ ]:
rfm['R_score'] = pd.qcut(rfm['Recency'], 4, duplicates='drop')
rfm['R_score'] = rfm['R_score'].astype('category').cat.codes

# balik urutan
rfm['R_score'] = rfm['R_score'].max() - rfm['R_score'] + 1

In [ ]:
rfm.head()

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score
Customer ID,,,,,,,
12346.0,56,10,230.55,2,3,1,331
12358.0,141,1,1429.83,1,1,4,414
12359.0,36,4,1522.23,3,3,4,234
12360.0,34,2,158.00,3,1,1,211
12361.0,91,1,109.20,1,1,1,411


In [ ]:
def segment(row):
    if row['R_score'] >= 3 and row['F_score'] >= 3:
        return 'Loyal Customer'
    elif row['R_score'] == 4:
        return 'Recent Customer'
    elif row['F_score'] == 1:
        return 'One-time Customer'
    else:
        return 'Potential Customer'

rfm['Segment'] = rfm.apply(segment, axis=1)

In [ ]:
rfm['Segment'].value_counts()

,count
Segment,
One-time Customer,1428
Recent Customer,370
Loyal Customer,367
Potential Customer,239


# MBA Analysis

In [ ]:
basket = df.groupby(['Invoice', 'Description'])['Quantity'] \
           .sum().unstack().fillna(0)

In [ ]:
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

/tmp/ipykernel_3185/3498954818.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


In [ ]:
!pip install mlxtend

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

frequent_itemsets = apriori(basket, min_support=0.02, use_colnames=True)

/usr/local/lib/python3.12/dist-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: Dep

In [ ]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

rules.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(PACK OF 60 PINK PAISLEY CAKE CASES),(60 TEATIME FAIRY CAKE CASES),0.049524,0.060952,0.024762,0.500000,8.203125,1.0,0.021743,1.878095,0.923848,0.288889,0.467546,0.453125
1,(60 TEATIME FAIRY CAKE CASES),(PACK OF 60 PINK PAISLEY CAKE CASES),0.060952,0.049524,0.024762,0.406250,8.203125,1.0,0.021743,1.600802,0.935091,0.288889,0.375313,0.453125
2,(PACK OF 72 RETRO SPOT CAKE CASES),(60 TEATIME FAIRY CAKE CASES),0.081270,0.060952,0.032063,0.394531,6.472778,1.0,0.027110,1.550943,0.920299,0.291066,0.355231,0.460286
3,(60 TEATIME FAIRY CAKE CASES),(PACK OF 72 RETRO SPOT CAKE CASES),0.060952,0.081270,0.032063,0.526042,6.472778,1.0,0.027110,1.938420,0.900388,0.291066,0.484116,0.460286
4,(72 SWEETHEART FAIRY CAKE CASES),(PACK OF 72 RETRO SPOT CAKE CASES),0.044762,0.081270,0.022063,0.492908,6.065076,1.0,0.018426,1.811762,0.874255,0.212214,0.448051,0.382196


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
rules = rules.sort_values(by='lift', ascending=False)

rules.head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
32,(SET/10 BLUE SPOTTY PARTY CANDLES),(SET/10 PINK SPOTTY PARTY CANDLES),0.023175,0.028730,0.020000,0.863014,30.038598,1.0,0.019334,7.090270,0.989644,0.626866,0.858962,0.779573
33,(SET/10 PINK SPOTTY PARTY CANDLES),(SET/10 BLUE SPOTTY PARTY CANDLES),0.028730,0.023175,0.020000,0.696133,30.038598,1.0,0.019334,3.214644,0.995305,0.626866,0.688924,0.779573
9,(RED 3 PIECE MINI DOTS CUTLERY SET),(BLUE 3 PIECE MINI DOTS CUTLERY SET),0.038413,0.035714,0.024762,0.644628,18.049587,1.0,0.023390,2.713455,0.982331,0.501608,0.631466,0.668981
8,(BLUE 3 PIECE MINI DOTS CUTLERY SET),(RED 3 PIECE MINI DOTS CUTLERY SET),0.035714,0.038413,0.024762,0.693333,18.049587,1.0,0.023390,3.135611,0.979582,0.501608,0.681083,0.668981
40,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.057460,0.035079,0.024603,0.428177,12.205945,1.0,0.022588,1.687446,0.974041,0.362150,0.407388,0.564767
41,(WOODEN PICTURE FRAME WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.035079,0.057460,0.024603,0.701357,12.205945,1.0,0.022588,3.156080,0.951449,0.362150,0.683151,0.564767
23,(LUNCH BAG RED SPOTTY),(LUNCH BAG BLACK SKULL.),0.063968,0.049524,0.027619,0.431762,8.718267,1.0,0.024451,1.672672,0.945799,0.321627,0.402154,0.494727
22,(LUNCH BAG BLACK SKULL.),(LUNCH BAG RED SPOTTY),0.049524,0.063968,0.027619,0.557692,8.718267,1.0,0.024451,2.116246,0.931426,0.321627,0.527465,0.494727
39,(WOODEN FRAME ANTIQUE WHITE ),(WOOD S/3 CABINET ANT WHITE FINISH),0.057460,0.043175,0.021111,0.367403,8.509709,1.0,0.018630,1.512536,0.936286,0.265469,0.338859,0.428187
38,(WOOD S/3 CABINET ANT WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.043175,0.057460,0.021111,0.488971,8.509709,1.0,0.018630,1.844394,0.922307,0.265469,0.457817,0.428187


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
rules.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
32,(SET/10 BLUE SPOTTY PARTY CANDLES),(SET/10 PINK SPOTTY PARTY CANDLES),0.023175,0.028730,0.020000,0.863014,30.038598,1.0,0.019334,7.090270,0.989644,0.626866,0.858962,0.779573
33,(SET/10 PINK SPOTTY PARTY CANDLES),(SET/10 BLUE SPOTTY PARTY CANDLES),0.028730,0.023175,0.020000,0.696133,30.038598,1.0,0.019334,3.214644,0.995305,0.626866,0.688924,0.779573
9,(RED 3 PIECE MINI DOTS CUTLERY SET),(BLUE 3 PIECE MINI DOTS CUTLERY SET),0.038413,0.035714,0.024762,0.644628,18.049587,1.0,0.023390,2.713455,0.982331,0.501608,0.631466,0.668981
8,(BLUE 3 PIECE MINI DOTS CUTLERY SET),(RED 3 PIECE MINI DOTS CUTLERY SET),0.035714,0.038413,0.024762,0.693333,18.049587,1.0,0.023390,3.135611,0.979582,0.501608,0.681083,0.668981
40,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.057460,0.035079,0.024603,0.428177,12.205945,1.0,0.022588,1.687446,0.974041,0.362150,0.407388,0.564767


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
rules = rules.sort_values(by='lift', ascending=False)

rules[['antecedents','consequents','support','confidence','lift']].head(10)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,support,confidence,lift
32,(SET/10 BLUE SPOTTY PARTY CANDLES),(SET/10 PINK SPOTTY PARTY CANDLES),0.020000,0.863014,30.038598
33,(SET/10 PINK SPOTTY PARTY CANDLES),(SET/10 BLUE SPOTTY PARTY CANDLES),0.020000,0.696133,30.038598
9,(RED 3 PIECE MINI DOTS CUTLERY SET),(BLUE 3 PIECE MINI DOTS CUTLERY SET),0.024762,0.644628,18.049587
8,(BLUE 3 PIECE MINI DOTS CUTLERY SET),(RED 3 PIECE MINI DOTS CUTLERY SET),0.024762,0.693333,18.049587
40,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.024603,0.428177,12.205945
41,(WOODEN PICTURE FRAME WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.024603,0.701357,12.205945
23,(LUNCH BAG RED SPOTTY),(LUNCH BAG BLACK SKULL.),0.027619,0.431762,8.718267
22,(LUNCH BAG BLACK SKULL.),(LUNCH BAG RED SPOTTY),0.027619,0.557692,8.718267
39,(WOODEN FRAME ANTIQUE WHITE ),(WOOD S/3 CABINET ANT WHITE FINISH),0.021111,0.367403,8.509709
38,(WOOD S/3 CABINET ANT WHITE FINISH),(WOODEN FRAME ANTIQUE WHITE ),0.021111,0.488971,8.509709


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Conclution

In [ ]:
rfm.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,Recency,Frequency,Monetary,R_score,F_score,M_score,RFM_Score,Segment
Customer ID,,,,,,,,
12346.0,56,10,230.55,2,3,1,331,Potential Customer
12358.0,141,1,1429.83,1,1,4,414,One-time Customer
12359.0,36,4,1522.23,3,3,4,234,Loyal Customer
12360.0,34,2,158.00,3,1,1,211,One-time Customer
12361.0,91,1,109.20,1,1,1,411,One-time Customer


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
rfm.to_csv("rfm.csv")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
rules.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
32,(SET/10 BLUE SPOTTY PARTY CANDLES),(SET/10 PINK SPOTTY PARTY CANDLES),0.023175,0.028730,0.020000,0.863014,30.038598,1.0,0.019334,7.090270,0.989644,0.626866,0.858962,0.779573
33,(SET/10 PINK SPOTTY PARTY CANDLES),(SET/10 BLUE SPOTTY PARTY CANDLES),0.028730,0.023175,0.020000,0.696133,30.038598,1.0,0.019334,3.214644,0.995305,0.626866,0.688924,0.779573
9,(RED 3 PIECE MINI DOTS CUTLERY SET),(BLUE 3 PIECE MINI DOTS CUTLERY SET),0.038413,0.035714,0.024762,0.644628,18.049587,1.0,0.023390,2.713455,0.982331,0.501608,0.631466,0.668981
8,(BLUE 3 PIECE MINI DOTS CUTLERY SET),(RED 3 PIECE MINI DOTS CUTLERY SET),0.035714,0.038413,0.024762,0.693333,18.049587,1.0,0.023390,3.135611,0.979582,0.501608,0.681083,0.668981
40,(WOODEN FRAME ANTIQUE WHITE ),(WOODEN PICTURE FRAME WHITE FINISH),0.057460,0.035079,0.024603,0.428177,12.205945,1.0,0.022588,1.687446,0.974041,0.362150,0.407388,0.564767


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
rules.to_csv("mba_rules.csv")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
df.to_csv("cleaned_data.csv", index=False)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag